# Notebook 05 — Cross-modal regional correspondence

Compares aperiodic parameters (exponent and offset) between iEEG and reconstructed  
sources at the region level. This is an **atlas-level** comparison — the datasets  
have different subjects and different recording conditions.

**Inputs**:
- `data/interim/region_aperiodic_ieeg.csv`
- `data/interim/region_aperiodic_sources.csv`

**Output**: Figure 4 — Cross-modal correspondence plots

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, pearsonr

PROJECT_ROOT = Path("../../").resolve()
INTERIM_DIR  = PROJECT_ROOT / "data" / "interim"
FIG_DIR      = PROJECT_ROOT / "manuscript" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
ieeg_region    = pd.read_csv(INTERIM_DIR / "region_aperiodic_ieeg.csv")
sources_region = pd.read_csv(INTERIM_DIR / "region_aperiodic_sources.csv")

print("iEEG regions:",   len(ieeg_region))
print("Source regions:", len(sources_region))

ieeg_region.head(3)

## 1. Match regions across modalities

Both datasets share the same Automated Anatomical Labeling (AAL) atlas with 38 regions.  
Inner join on `Region name` retains only regions present in both datasets.

In [ ]:
merged = ieeg_region.merge(
    sources_region,
    on=["Region name", "Lobe"],
    suffixes=("_ieeg", "_sources"),
)

print(f"Matched regions: {len(merged)} / {len(ieeg_region)} iEEG, {len(sources_region)} sources")

# Regions in iEEG but not sources (coverage gaps)
only_ieeg = set(ieeg_region["Region name"]) - set(sources_region["Region name"])
only_src  = set(sources_region["Region name"]) - set(ieeg_region["Region name"])
print(f"Only in iEEG ({len(only_ieeg)}): {sorted(only_ieeg)[:5]} …"  if only_ieeg else "No iEEG-only regions.")
print(f"Only in sources ({len(only_src)}): {sorted(only_src)[:5]} …" if only_src  else "No source-only regions.")

merged.head(3)

## 2. Correspondence statistics

In [ ]:
def report_correspondence(x, y, label):
    """Report Spearman and Pearson r for paired region values."""
    mask = ~(np.isnan(x) | np.isnan(y))
    x, y = x[mask], y[mask]
    rho, p_spear = spearmanr(x, y)
    r,   p_pear  = pearsonr(x, y)
    print(f"{label}: n={len(x)}, Spearman ρ={rho:.3f} (p={p_spear:.4f}),"
          f" Pearson r={r:.3f} (p={p_pear:.4f})")
    return rho, p_spear

rho_exp, p_exp = report_correspondence(
    merged["exponent_median_ieeg"].values,
    merged["exponent_median_sources"].values,
    "Exponent"
)
rho_off, p_off = report_correspondence(
    merged["offset_median_ieeg"].values,
    merged["offset_median_sources"].values,
    "Offset"
)

## 3. Figure 4 — Cross-modal scatter plots

In [ ]:
LOBE_COLORS = {
    "Frontal":   "#E64B35",
    "Temporal":  "#4DBBD5",
    "Parietal":  "#00A087",
    "Occipital": "#3C5488",
    "Limbic":    "#F39B7F",
    "Insula":    "#8491B4",
    "Other":     "#91D1C2",
}


def crossmodal_scatter(ax, x, y, labels, lobes, param, rho, p):
    colors = [LOBE_COLORS.get(str(l), LOBE_COLORS["Other"]) for l in lobes]
    ax.scatter(x, y, c=colors, s=60, alpha=0.85, edgecolors="white", linewidths=0.5)

    # Annotate regions that deviate most
    residuals = np.abs(y - x)
    top_idx   = np.argsort(residuals)[-4:]
    for i in top_idx:
        ax.annotate(
            labels[i], (x[i], y[i]),
            fontsize=6, xytext=(4, 4), textcoords="offset points"
        )

    # Unity line
    lims = [min(x.min(), y.min()) - 0.1, max(x.max(), y.max()) + 0.1]
    ax.plot(lims, lims, "k--", linewidth=0.8, alpha=0.5, label="unity")

    ax.set_xlabel(f"iEEG {param} (median)")
    ax.set_ylabel(f"Sources {param} (median)")
    ax.set_title(f"{param.capitalize()} — Spearman ρ = {rho:.2f} (p = {p:.3f})")

    # Lobe legend
    legend_patches = [
        plt.Rectangle((0, 0), 1, 1, fc=c, label=lobe)
        for lobe, c in LOBE_COLORS.items()
        if lobe in lobes.values
    ]
    ax.legend(handles=legend_patches, fontsize=7, loc="upper left")


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

crossmodal_scatter(
    ax1,
    x      = merged["exponent_median_ieeg"].values,
    y      = merged["exponent_median_sources"].values,
    labels = merged["Region name"].values,
    lobes  = merged["Lobe"],
    param  = "exponent",
    rho    = rho_exp, p = p_exp,
)

crossmodal_scatter(
    ax2,
    x      = merged["offset_median_ieeg"].values,
    y      = merged["offset_median_sources"].values,
    labels = merged["Region name"].values,
    lobes  = merged["Lobe"],
    param  = "offset",
    rho    = rho_off, p = p_off,
)

fig.suptitle("Figure 4 — Cross-modal regional correspondence\n"
             "(atlas-level; different subjects across modalities)",
             fontsize=12)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig4_crossmodal_correspondence.svg", bbox_inches="tight", dpi=150)
plt.show()

## 4. Region-level correspondence table

Ranks regions by the absolute discrepancy between modalities.

In [ ]:
table = merged[["Region name", "Lobe",
                "exponent_median_ieeg", "exponent_median_sources",
                "offset_median_ieeg",   "offset_median_sources"]].copy()

table["delta_exponent"] = table["exponent_median_sources"] - table["exponent_median_ieeg"]
table["delta_offset"]   = table["offset_median_sources"]   - table["offset_median_ieeg"]
table["abs_delta_exp"]  = table["delta_exponent"].abs()

print("Regions with largest exponent discrepancy (iEEG vs Sources):")
display(table.sort_values("abs_delta_exp", ascending=False).head(10).round(3))

print("\nRegions with smallest exponent discrepancy (best correspondence):")
display(table.sort_values("abs_delta_exp", ascending=True).head(10).round(3))

# Save
table.to_csv(INTERIM_DIR / "crossmodal_region_comparison.csv", index=False)
print("\nSaved crossmodal table.")